In [1]:
import sys
import os


sys.path.append(os.path.abspath(".."))

In [2]:
import gradio as gr
import json
from datetime import datetime
from pymongo import MongoClient
from langchain_core.messages import HumanMessage

from graph.workflow import app
from utils.checklist_format import checklist_format
from utils.generate_pdf import generate_pdf

c:\Users\Lenovo\Desktop\QualiChainAI\Backend\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 19929.91it/s]


In [3]:
client = MongoClient("mongodb://localhost:27017/")

db = client["qualichainAI"]

checklists_collection = db["audit_checklists"]
reports_collection = db["audit_reports"]
capa_collection = db["capa_plans"]

In [4]:
AUDIT_TYPES = [
    "Audit transport pharmaceutique",
    "Audit système qualité",
    "Audit conformité réglementaire",
    "Audit fournisseur pharmaceutique",
    "Audit entrepôt de stockage pharmaceutique",
    "Audit distributeur pharmaceutique",
    "Audit chaîne du froid pharmaceutique"
]


SITES_PHARMACEUTIQUES = [

    "Dépôt central Tunis",
    "Entrepôt Ariana",
    "Centre de distribution Sousse",
    "Plateforme logistique Sfax",
    "Entrepôt frigorifique Monastir",
    "Grossiste répartiteur Nord",
    "Grossiste répartiteur Sud",
    "Sous-traitant transport pharmaceutique",
    "Fournisseur médicaments",
    "Fournisseur dispositifs médicaux",
    "Site de stockage vaccins",
    "Pharmacie hospitalière",
    "Centre de distribution régional",
    "Entrepôt produits thermosensibles",
    "Prestataire logistique GDP"

]

In [5]:
def generate_checklist(type_audit, site_audit):

    if not type_audit:
        yield "Veuillez sélectionner un type d'audit.", None
        return

    if not site_audit:
        yield "Veuillez sélectionner un site.", None
        return

    yield "⏳ Checklist en cours de génération...", None

    result = app.invoke(
        {
            "messages": [
                HumanMessage(
                    content=(
                        f"Génère une checklist pour un audit "
                        f"de type {type_audit} "
                        f"pour le site {site_audit}"
                    )
                )
            ]
        }
    )

    content = result["messages"][-1].content

    try:

        checklist = json.loads(content)

    except json.JSONDecodeError:

        yield "❌ Erreur : la checklist retournée n'est pas un JSON valide.", None
        return

    checklist = checklist_format(checklist)

    checklist_id = (
        f"CHK-{checklists_collection.count_documents({}) + 1:03d}"
    )

    checklist = {
        "checklist_id": checklist_id,
        **checklist
    }


    checklists_collection.insert_one(checklist)
    yield (
        f"✅ Checklist {checklist_id} générée avec succès.",
        checklist
    )



In [6]:
def save_completed_checklist(checklist,responsable,date_audit,*values):

    if not checklist:
        return "❌ Aucune checklist chargée."

    try:
        date_audit = datetime.strptime(date_audit, "%d/%m/%Y")
    except (ValueError, TypeError):
        return "❌ Format de date invalide. Utilisez JJ/MM/AAAA."
    
    checklist["responsable"] = responsable or ""
    checklist["date_audit"] = date_audit

    index = 0

    for section in checklist.get("sections", []):

        for point in section.get("points_controle", []):

            resultat = values[index]
            commentaire = values[index + 1]

            if resultat == "Oui":
                point["resultat"] = "Conforme"

            elif resultat == "Non":
                point["resultat"] = "Non conforme"

            else:
                point["resultat"] = ""


            point["commentaire"] = commentaire or ""

            index += 2

    checklists_collection.update_one(
        {
            "checklist_id": checklist["checklist_id"]
        },
        {
            "$set": {
                "date_audit": checklist["date_audit"],
                "responsable": checklist["responsable"],
                "sections": checklist["sections"],
                "status": "remplie"
            }
        }
    )

    return (
        f"✅ Checklist {checklist['checklist_id']} "
        f"enregistrée avec succès."
    )

In [7]:
def export_checklist_pdf(checklist):

    if not checklist:
        raise Exception("Aucune checklist chargée")

    document = checklists_collection.find_one(
        {
            "checklist_id": checklist["checklist_id"]
        }
    )

    if not document:
        raise Exception("Checklist introuvable")

    output_path = (
        f"checklists/{checklist['checklist_id']}.pdf"
    )

    generate_pdf(
        data=document,
        output_file=output_path,
        title=f"Checklist {checklist['checklist_id']}"
    )

    return output_path

In [8]:
def analyze_checklist_ui(checklist):

    if not checklist:
        yield "❌ Aucune checklist chargée."
        return

    yield "⏳ Analyse en cours..."

    result = app.invoke(
        {
            "messages": [
                HumanMessage(
                    content=f"Analyse la checklist {checklist['checklist_id']}"
                )
            ]
        }
    )

    try:
        analysis = json.loads(
            result["messages"][-1].content
        )

    except Exception:
        yield "❌ L'analyse retournée n'est pas un JSON valide."
        return

    checklists_collection.update_one(
        {
            "checklist_id": checklist["checklist_id"]
        },
        {
            "$set": {
                "analysis": analysis
            }
        }
    )

    observations = analysis.get("observations", [])
    obs_md = "\n".join(f"- {obs}"for obs in observations ) if observations else "Aucune observation."


    yield f"""
## 📊 Analyse de la checklist

### 🎯 Score de conformité

**{analysis.get("score_conformite", 0)} %**

### 📌 Statut global

**{analysis.get("statut_global", "Non défini")}**

### 📋 Résumé

- **Total des points :** {analysis.get("resume", {}).get("total_points", 0)}
- **Conformes :** {analysis.get("resume", {}).get("conformes", 0)}
- **Non conformes :** {analysis.get("resume", {}).get("non_conformes", 0)}
- **Partiellement conformes :** {analysis.get("resume", {}).get("partiellement_conformes", 0)}
- **Non applicables :** {analysis.get("resume", {}).get("non_applicables", 0)}

### 👁️ Observations

{obs_md}

"""


In [9]:
def generate_capa_ui(checklist):

    if not checklist:
        return "❌ Aucune checklist chargée."

    yield "⏳ Génération du plan CAPA en cours..."

    result = app.invoke(
        {
            "messages": [
                HumanMessage(
                    content=(
                        f"Génère un plan CAPA "
                        f"pour la checklist "
                        f"{checklist['checklist_id']}"
                    )
                )
            ]
        }
    )
    try:
        capa = json.loads(
            result["messages"][-1].content
        )

    except Exception:

        yield "❌ Le CAPA retourné n'est pas un JSON valide."
        return


    capa_document = {
        "checklist_id": checklist["checklist_id"],
        **capa
    }

    capa_collection.insert_one(
        capa_document
    )

    markdown = f"""
    # 🛠️ Plan CAPA
    
    **Checklist :** {checklist['checklist_id']}
    
    **Nombre d'actions :** {len(capa.get('actions', []))}
    """
    for i, action in enumerate(
            capa.get("actions", []),
            start=1
        ):
    
            markdown += f"""
    
    ---
    
    ## Action {i}
    
    ### 🚨 Problème
    {action.get("probleme", "N/A")}
    
    ### 🔍 Cause racine
    {action.get("cause_racine", "N/A")}
    
    ### ✅ Action corrective
    {action.get("action_corrective", "N/A")}
    
    ### 🛡️ Action préventive
    {action.get("action_preventive", "N/A")}
    
    ### ⚠️ Priorité
    **{action.get("priorite", "N/A")}**
    
    ### 👤 Responsable
    {action.get("responsable", "N/A")}
    
    ### 📅 Échéance
    {action.get("echeance", "N/A")}
    
    ### 📌 Statut
    {action.get("statut_action", "N/A")}
    """
    
    yield markdown



In [10]:
def generate_report_ui(checklist):

    if not checklist:
        return "❌ Aucune checklist chargée."

    result = app.invoke(
        {
            "messages": [
                HumanMessage(
                    content=f"Génère un rapport pour la checklist {checklist['checklist_id']}"
                )
            ]
        }
    )

    content = result["messages"][-1].content
    print("=" * 100)
    print("RÉPONSE AGENT :")
    print(content)
    print("=" * 100)


    try:
        report = json.loads(content)

    except json.JSONDecodeError:
            print("❌ La réponse de l'agent n'est pas un JSON valide.")
            print("Contenu reçu :", repr(content))

            return "❌ Le rapport généré n'est pas au format JSON valide."
    report_document = {
        "checklist_id": checklist["checklist_id"],
        **report
    }
    # sauvegarde MongoDB
    reports_collection.insert_one(report_document)

    document = reports_collection.find_one(
        {
            "checklist_id": checklist["checklist_id"]
        }
    )

    if not document:
        raise Exception("Rapport introuvable")

    output_path = (
        f"reports/{checklist['checklist_id']}_report.pdf"
    )
    generate_pdf(
        data=document,
        output_file=output_path,
        title=f"Rapport d'audit {checklist['checklist_id']} "
    )

    return output_path

In [11]:
from agents import regulatory_agent
def chat(message, history):

    result = regulatory_agent.invoke(
        {
            "messages": [
                HumanMessage(content=message)
            ]
        }
    )

    return result["messages"][-1].content

In [13]:
with gr.Blocks(title="QualiChain AI") as demo:

    checklist_state = gr.State(None)

    # =====================================================
    # ASSISTANT
    # =====================================================

    with gr.Tab("💬 Assistant QualiChain AI"):

        gr.ChatInterface(
            fn=chat,
            chatbot=gr.Chatbot(
                label="Assistant"
            ),
            textbox=gr.Textbox(
                placeholder="Posez votre question..."
            ),
            title="🤖 QualiChain AI"
        )

    # =====================================================
    # AUDIT
    # =====================================================

    with gr.Tab("📋 Audit"):

        with gr.Tabs():

            # =================================================
            # ONGLET CHECKLIST
            # =================================================

            with gr.Tab("1️⃣ Checklist"):

                gr.Markdown(
                    "# 📋 Création d'une checklist d'audit"
                )

                with gr.Row():

                    type_audit = gr.Dropdown(
                        choices=AUDIT_TYPES,
                        label="Type d'audit"
                    )

                    site_audit = gr.Dropdown(
                        choices=SITES_PHARMACEUTIQUES,
                        label="Site audité",
                        allow_custom_value=True
                    )

                generate_button = gr.Button(
                    "🚀 Créer checklist",
                    variant="primary"
                )

                generation_status = gr.Markdown()

                # =================================================
                # CHECKLIST DYNAMIQUE
                # =================================================

                @gr.render(inputs=checklist_state)
                def render_checklist(checklist):

                    if not checklist:
                        return

                    gr.Markdown(
                        f"""
                        ## 📋 Checklist : {checklist['checklist_id']}

                        **Type :** {checklist['type_audit']}  

                        **Site :** {checklist['site_audit']}
                        """
                    )
                    
                    date_audit = gr.Textbox(label="Date de l'audit",placeholder="JJ/MM/AAAA")


                    responsable = gr.Textbox(
                        label="Responsable de l'audit",
                        placeholder="Nom du responsable..."
                    )

                    dynamic_values = []

                    # =================================================
                    # SECTIONS
                    # =================================================

                    for section in checklist.get(
                        "sections",
                        []
                    ):

                        points = section.get(
                            "points_controle",
                            []
                        )

                        gr.Markdown(
                            f"""
                            ## 📂 {section['nom_section']}

                            **Nombre de questions : {len(points)}**
                            """
                        )

                        # =================================================
                        # QUESTIONS
                        # =================================================

                        for i, point in enumerate(
                            points,
                            start=1
                        ):

                            gr.Markdown(
                                f"""
                                **{i}. {point['question']}**

                                **Criticité :**
                                `{point['criticite']}`

                                **📑 Preuve attendue :**
                                {point.get(
                                    'preuve_attendue',
                                    'Non spécifiée'
                                )}
                                """
                            )

                            resultat = gr.Radio(
                                choices=[
                                    "Oui",
                                    "Non"
                                ],
                                label="Résultat",
                                value=None
                            )

                            commentaire = gr.Textbox(
                                label="Commentaire",
                                placeholder="Ajouter un commentaire..."
                            )

                            dynamic_values.extend(
                                [
                                    resultat,
                                    commentaire
                                ]
                            )

                            gr.Markdown("---")

                    # =================================================
                    # ENREGISTRER
                    # =================================================

                    save_btn = gr.Button(
                        "💾 Enregistrer checklist",
                        variant="primary"
                    )

                    save_status = gr.Markdown()

                    save_btn.click(
                        save_completed_checklist,
                        inputs=[
                            checklist_state,
                            responsable,
                            date_audit,
                            *dynamic_values
                        ],
                        outputs=save_status
                    )

                    # =================================================
                    # EXPORT PDF
                    # =================================================

                    gr.Markdown(
                        "### 📄 Export de la checklist"
                    )

                    export_checklist_btn = gr.Button(
                        "📄 Exporter la checklist en PDF"
                    )

                    checklist_pdf = gr.File(
                        label="Checklist PDF"
                    )

                    export_checklist_btn.click(
                        fn=export_checklist_pdf,
                        inputs=[
                            checklist_state
                        ],
                        outputs=[
                            checklist_pdf
                        ]
                    )

            # =================================================
            # ONGLET ANALYSE
            # =================================================

            with gr.Tab("2️⃣ Analyse"):

                gr.Markdown(
                    "# 🔍 Analyse de la checklist"
                )

                analyze_btn = gr.Button(
                    "🔍 Analyser checklist",
                    variant="primary"
                )

                analysis_box = gr.Markdown()

            # =================================================
            # ONGLET CAPA
            # =================================================

            with gr.Tab("3️⃣ CAPA"):

                gr.Markdown(
                    "# 🛠️ Plan CAPA"
                )

                generate_capa_btn = gr.Button(
                    "🛠️ Générer CAPA",
                    variant="primary"
                )

                capa_box = gr.Markdown()

            # =================================================
            # ONGLET RAPPORT
            # =================================================

            with gr.Tab("4️⃣ Rapport"):

                gr.Markdown(
                    "# 📑 Rapport d'audit"
                )

                generate_report_btn = gr.Button(
                    "📑 Générer rapport",
                    variant="primary"
                )

                report_pdf = gr.File(
                    label="Rapport PDF"
                )

    # =====================================================
    # EVENTS
    # =====================================================

    generate_button.click(
        generate_checklist,
        inputs=[
            type_audit,
            site_audit
        ],
        outputs=[
            generation_status,
            checklist_state
        ]
    )

    analyze_btn.click(
        analyze_checklist_ui,
        inputs=[
            checklist_state
        ],
        outputs=[
            analysis_box
        ]
    )

    generate_capa_btn.click(
        generate_capa_ui,
        inputs=[
            checklist_state
        ],
        outputs=[
            capa_box
        ]
    )

    generate_report_btn.click(
        generate_report_ui,
        inputs=[
            checklist_state
        ],
        outputs=[
            report_pdf
        ]
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "c:\Users\Lenovo\Desktop\QualiChainAI\Backend\venv\Lib\site-packages\gradio\queueing.py", line 963, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "c:\Users\Lenovo\Desktop\QualiChainAI\Backend\venv\Lib\site-packages\gradio\route_utils.py", line 409, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<12 lines>...
    )
    ^
  File "c:\Users\Lenovo\Desktop\QualiChainAI\Backend\venv\Lib\site-packages\gradio\blocks.py", line 2316, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<9 lines>...
    )
    ^
  File "c:\Users\Lenovo\Desktop\QualiChainAI\Backend\venv\Lib\site-packages\gradio\blocks.py", line 1695, in call_function
    prediction = await utils.async_iteration(iterator)
                 ^^^^^^^^^^^^^^^^^^^